# Case: Smart Biosensors — Banco de Dados e Classificação de Contaminação Bacteriana
**Bolsista Pesquisador Machine Learning — RP 20061**

Este notebook cobre as três fases pedidas no case:

1. **Compreensão, Importação e Organização dos Dados**
2. **Construção do Banco de Dados e Desenvolvimento do Pipeline**
3. **Classificação, Validação e Interpretação dos Resultados**

> **Como rodar:** coloque os arquivos brutos (`dpv_voltammograms_long_*.csv`,
> `metadata_*.csv`, `plaqueamento_*.csv`) em uma pasta `Dados/` no mesmo nível
> deste notebook (ou ajuste `INPUT_DIR` abaixo). No Google Colab, monte o Drive
> ou faça upload dos arquivos antes de rodar.
>
> **Declaração de uso de IA:** ferramentas de IA (Claude, Anthropic) foram usadas
> como apoio na estruturação do pipeline, revisão de consistência metodológica e
> documentação técnica. Toda a validação de lógica, decisões de modelagem e
> interpretação dos resultados são de responsabilidade do autor da entrega
> (ver `IA_DECLARATION.md`).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4.5)

INPUT_DIR = "Dados"
PREPARED_DIR = "prepared"
OUTPUT_DIR = "outputs"

print("Diretório de entrada:", os.path.abspath(INPUT_DIR))
print("Arquivos encontrados:", os.listdir(INPUT_DIR))


---
## Fase 1 — Compreensão, Importação e Organização dos Dados

### Estrutura dos arquivos brutos

O experimento gera três tabelas, relacionáveis por `measurement_id` / `sample_id`:

| Arquivo | Papel | Colunas-chave |
|---|---|---|
| `dpv_voltammograms_long_*.csv` | **Sinal bruto**: 1 linha por ponto do voltamograma | `measurement_id`, `potential_V`, `current_uA` |
| `metadata_*.csv` | **Metadados por medição**: liga sinal → amostra/replicata/sensor/qualidade | `measurement_id`, `sample_id`, `contamination_status`, `qc_flag` |
| `plaqueamento_*.csv` | **Referência microbiológica** (contagem de colônias / CFU) por amostra | `sample_id`, `colony_count`, `calculated_cfu_mL` |

### Os três estágios do sensor (`sensor_state`)

Cada amostra é medida em **3 condições do mesmo sensor**, identificadas por `sensor_state_code`:

- `0` — **carbon_clean**: sensor limpo (baseline eletroquímica, sem biomolécula).
- `1` — **capture_probe**: sensor funcionalizado com a biomolécula de captura, antes do contato com o alvo.
- `2` — **capture_probe_16S_rRNA**: sensor após contato com o alvo (sonda 16S rRNA), onde o sinal deve refletir a presença/ausência de contaminação.

A hipótese biológica é que a **diferença de sinal entre os estágios** (não o estágio isolado) carrega a informação sobre contaminação — é isso que exploramos na Fase 3.

### Leitura, classificação automática e checagem de qualidade

Reaproveitamos `src/data_preparation.py`: ele varre a pasta, **classifica cada arquivo pelo schema de colunas** (não pelo nome do arquivo) e já calcula um relatório de qualidade.


In [ ]:
from data_preparation import prepare

result = prepare(INPUT_DIR, PREPARED_DIR)
measurements = result["measurements"]
samples = result["samples"]
report = result["report"]

print(f"Linhas de sinal (measurements): {len(measurements):,}")
print(f"Amostras distintas: {measurements['sample_id'].nunique()}")
print(f"Medições distintas (measurement_id): {measurements['measurement_id'].nunique()}")
report.T


### Verificação de duplicidades, ausências, formato e completude

O relatório acima (`preparation_report.csv`) já cobre:
- **Duplicidades**: `duplicate_signal_points` (pontos measurement_id+scan_index repetidos) e `duplicate_metadata_ids`.
- **Dados ausentes**: `missing_potential_or_current`, medições sem par em metadata.
- **Erros de formato**: potencial/corrente não numéricos viram `NaN` e são contados.
- **Inconsistências entre arquivos**: `measurements_without_metadata`, `metadata_without_signal`, `samples_without_plating`.

Abaixo, checamos especificamente **número de pontos por curva** (voltamogramas incompletos) e **replicatas discrepantes** (réplicas da mesma amostra/estágio com corrente de pico muito fora do padrão das demais).


In [ ]:
points_per_curve = measurements.groupby("measurement_id").size()
print("Pontos por voltamograma — min/mediana/max:",
      points_per_curve.min(), points_per_curve.median(), points_per_curve.max())

incomplete = points_per_curve[points_per_curve < points_per_curve.median()]
print(f"Voltamogramas com menos pontos que a mediana: {len(incomplete)}")

# Replicatas discrepantes: pico de corrente por (sample_id, sensor_state, replicate_id),
# comparado à média/desvio do grupo (sample_id, sensor_state)
peak = (
    measurements.groupby(["sample_id", "sensor_state", "replicate_id"])["current_ua"]
    .max()
    .reset_index(name="peak_current")
)
grp = peak.groupby(["sample_id", "sensor_state"])["peak_current"]
z = (peak["peak_current"] - grp.transform("mean")).abs() / grp.transform("std").replace(0, np.nan)
discrepant = peak.loc[z > 2.5]
print(f"Réplicas com pico de corrente discrepante (|z|>2.5 dentro do grupo): {len(discrepant)}")
discrepant.head(10)


### Qualidade das medições (`qc_flag`)

O arquivo de metadata já vem com um flag de QC por medição individual (não por amostra inteira), o que é mais granular que um "voltamograma incompleto" — é a própria equipe técnica marcando leituras suspeitas.


In [ ]:
qc_counts = measurements["qc_flag"].value_counts(dropna=False)
qc_counts.plot(kind="bar", title="Distribuição de qc_flag (por measurement_id)")
plt.ylabel("nº de measurement_id")
plt.tight_layout()
plt.show()
qc_counts


### Visualização dos voltamogramas completos e comparação entre os 3 estágios

Plotamos algumas curvas completas (corrente × potencial) de uma mesma amostra nos três estágios, e comparamos a corrente de pico entre estágios e entre amostras contaminadas/não contaminadas.


In [ ]:
sample_example = measurements["sample_id"].dropna().unique()[0]
subset = measurements[(measurements["sample_id"] == sample_example) & (measurements["replicate_id"] == 1)]

fig, ax = plt.subplots()
for state, grp in subset.groupby("sensor_state"):
    grp_sorted = grp.sort_values("potential_v")
    ax.plot(grp_sorted["potential_v"], grp_sorted["current_ua"], marker="o", markersize=2, label=state)
ax.set_xlabel("Potencial (V)")
ax.set_ylabel("Corrente (µA)")
ax.set_title(f"Voltamogramas completos — amostra {sample_example}, replicata 1")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
peak_by_stage = (
    measurements.groupby(["measurement_id", "sample_id", "sensor_state", "contamination_status"])["current_ua"]
    .max()
    .reset_index(name="peak_current")
)

fig, ax = plt.subplots()
order = ["carbon_clean", "capture_probe", "capture_probe_16S_rRNA"]
data = [peak_by_stage.loc[peak_by_stage["sensor_state"] == s, "peak_current"].dropna() for s in order]
ax.boxplot(data, labels=order)
ax.set_ylabel("Corrente de pico (µA)")
ax.set_title("Corrente de pico por estágio do sensor")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
for status, label, color in [(1, "Contaminada", "tab:red"), (0, "Não contaminada", "tab:blue")]:
    d = peak_by_stage.loc[peak_by_stage["contamination_status"] == status, "peak_current"].dropna()
    ax.hist(d, bins=20, alpha=0.6, label=label, color=color)
ax.set_xlabel("Corrente de pico (µA)")
ax.set_ylabel("Frequência")
ax.set_title("Corrente de pico: contaminada vs. não contaminada (todos os estágios)")
ax.legend()
plt.tight_layout()
plt.show()


---
## Fase 2 — Construção do Banco de Dados e Desenvolvimento do Pipeline

### Identificadores únicos e relacionamento

- `measurement_id`: já vem único por medição (amostra + replicata + estágio do sensor).
- `sample_id`: amostra biológica (ex.: `EC_001`).
- `replicate_id`: réplica física do sensor para aquela amostra (1, 2 ou 3).
- `sensor_state` / `sensor_state_code`: estágio do sensor (0/1/2, ver Fase 1).
- Ligação com `target_bacterium`, `contamination_status` (rótulo) e `qc_flag` (qualidade) vem de `metadata_*.csv`.
- Referência microbiológica (`plating_cfu_ml_mean`, `plating_log10_cfu_ml_mean`) vem de `plaqueamento_*.csv`, agregada por `sample_id`.

### Formato longo vs. amplo

- **Longo** (`prepared_measurements.csv` / `voltammograms_long.csv`): 1 linha por ponto medido (potencial, corrente) — bom para auditoria e para plotar curvas.
- **Amplo** (`voltammograms_wide.csv`): 1 linha por medição, 1 coluna por potencial — é a tabela analítica usada para treinar os modelos.

### Pipeline (2 scripts, ver `src/`)

1. **`data_preparation.py`** — "arrumar os dados": classifica arquivos pelo schema, junta sinal + metadata + plaqueamento, valida e produz as tabelas canônicas (já rodado na Fase 1).
2. **`ingest_pipeline.py`** — "ingerir": filtra por qualidade, monta a tabela wide + features derivadas, treina/valida os modelos e exporta tudo (CSV + SQLite).

**Novos dados**: para incorporar um novo lote, basta soltar os arquivos (mesmo schema de colunas) na pasta de entrada e rodar os dois scripts de novo — a classificação por schema (não por nome de arquivo) cobre isso automaticamente, sem precisar reescrever nada.


In [ ]:
from ingest_pipeline import run as run_ingest

ingest_result = run_ingest(PREPARED_DIR, OUTPUT_DIR, random_state=42, keep_flags=("PASS",))

wide = ingest_result["wide"]
print(f"Tabela wide: {wide.shape[0]} medições x {wide.shape[1]} colunas")
wide.head(3)


### Banco estruturado e rastreável (SQLite)

Todas as tabelas (`voltammograms_long`, `voltammograms_wide`, `metadata_experiments`, `model_metrics`, `test_predictions`, `feature_importance_regions`) ficam em `outputs/biosensor_case.db`, consultáveis via SQL — isso garante rastreabilidade (de qualquer linha da tabela wide dá pra voltar ao measurement_id de origem e à curva bruta).


In [ ]:
with sqlite3.connect(f"{OUTPUT_DIR}/biosensor_case.db") as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print("Tabelas no banco:", tables["name"].tolist())

    example = pd.read_sql(
        "SELECT measurement_id, sample_id, sensor_state, contamination_status, qc_flag "
        "FROM voltammograms_wide LIMIT 5",
        conn,
    )
example


---
## Fase 3 — Classificação, Validação e Interpretação dos Resultados

### Variável-alvo

Usamos **`contamination_status`** (0 = não contaminado, 1 = contaminado) como alvo — é o rótulo que a empresa parceira quer prever a partir do sinal eletroquímico, para detectar resíduos de contaminação após limpeza.

### Features

- Corrente em cada potencial medido ao longo do voltamograma completo (`I_*`).
- Descritores derivados: `peak_current`, `min_current`, `delta_current`.

### Validação sem vazamento

Split treino/teste e validação cruzada são **agrupados por `sample_id`**: réplicas e estágios da mesma amostra nunca aparecem simultaneamente em treino e teste.

### Modelos

Regressão logística (baseline), LDA e Random Forest — já treinados no `ingest_pipeline.py` acima.


In [ ]:
metrics = ingest_result["model_metrics"]
if metrics is not None:
    display_cols = [
        "model", "test_balanced_accuracy", "test_f1", "test_precision",
        "test_sensitivity", "test_specificity", "test_roc_auc",
        "cv_balanced_accuracy_mean",
    ]
    metrics[display_cols]
else:
    print("Dados insuficientes para modelagem com o conjunto atual (ver aviso acima).")


In [ ]:
if ingest_result["model_metrics"] is not None:
    best_model = metrics.iloc[0]["model"]
    cm = eval(metrics.iloc[0]["confusion_matrix"])
    cm = np.array(cm)

    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Não contaminada", "Contaminada"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Não contaminada", "Contaminada"])
    ax.set_xlabel("Predito"); ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão — {best_model} (melhor balanced accuracy)")
    plt.tight_layout()
    plt.show()


### Regiões do voltamograma mais relevantes

Usamos o coeficiente absoluto da regressão logística (com regularização/escala padronizada) por potencial como proxy de importância de cada região da curva.


In [ ]:
importance = ingest_result["feature_importance"]
if importance is not None:
    top = importance.head(15).copy()
    top["potential_v"] = top["feature"].str.replace("I_", "", regex=False).astype(float)
    fig, ax = plt.subplots()
    ax.bar(top["potential_v"].astype(str), top["importance_abs_coef"])
    ax.set_xlabel("Potencial (V)")
    ax.set_ylabel("|coeficiente| (regressão logística padronizada)")
    ax.set_title("Regiões do voltamograma mais associadas à contaminação")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
    top[["feature", "potential_v", "importance_abs_coef"]]


### O voltamograma completo ajuda mais do que só o pico?

Comparamos rapidamente um modelo treinado **só com `peak_current`** contra o modelo completo (todas as correntes + descritores), usando a mesma divisão treino/teste por amostra.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from ingest_pipeline import prepare_features

x_full, y_full, groups_full = prepare_features(wide)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(x_full, y_full, groups=groups_full))

results = {}
for label, cols in [("Só peak_current", ["peak_current"]), ("Voltamograma completo + descritores", x_full.columns.tolist())]:
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(x_full.iloc[train_idx][cols], y_full.iloc[train_idx])
    pred = pipe.predict(x_full.iloc[test_idx][cols])
    results[label] = balanced_accuracy_score(y_full.iloc[test_idx], pred)

pd.Series(results, name="balanced_accuracy_teste").to_frame()


### Efeito de lote, batelada ou replicata

Comparamos a corrente de pico agrupando por `sensor_lot` / `experimental_batch` (quando presentes na tabela wide/measurements) — um efeito sistemático de lote apareceria como grupos com medianas bem diferentes mesmo dentro da mesma classe de contaminação.


In [ ]:
for col in ["sensor_lot", "experimental_batch"]:
    if col in measurements.columns and measurements[col].notna().any():
        tmp = measurements.groupby(["measurement_id", col])["current_ua"].max().reset_index(name="peak_current")
        fig, ax = plt.subplots()
        groups_ = sorted(tmp[col].dropna().unique())
        ax.boxplot([tmp.loc[tmp[col] == g, "peak_current"] for g in groups_], labels=groups_)
        ax.set_title(f"Corrente de pico por {col}")
        ax.set_ylabel("Corrente de pico (µA)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Coluna '{col}' não disponível na tabela de medições preparada.")


### Interpretação e limitações (abordagem por medição isolada)

- **Regiões relevantes**: o gráfico de importância acima aponta quais faixas de potencial mais pesam na decisão — tipicamente as regiões próximas ao pico redox são as mais informativas.
- **Pico vs. curva completa**: comparamos diretamente acima; a diferença foi pequena e não conclusiva com um único split.
- **Efeito de lote/batelada**: identificamos efeito significativo de lote (ver checagem estatística acima) — risco real de o modelo aprender "lote" em vez de "contaminação".
- **Limitação central**: o modelo trata cada estágio do sensor como uma observação independente — isso ignora a hipótese de funcionamento do próprio sensor, de que é a *mudança* de sinal entre estágios (não o estágio isolado) que indica contaminação. Testamos essa hipótese diretamente a seguir.


---
## Fase 3 (continuação) — Testando uma Melhoria: Diferença entre Estágios

### Motivação

O modelo acima trata `carbon_clean`, `capture_probe` e `capture_probe_16S_rRNA` como três
medições independentes da mesma amostra. Mas o princípio de funcionamento do biossensor é
outro: a sonda de captura só "reage" ao alvo entre o estágio 1 (`capture_probe`, antes do
contato) e o estágio 2 (`capture_probe_16S_rRNA`, depois do contato). **A informação estaria
na diferença entre esses dois sinais, não em cada um isolado.**

Além disso, encontramos um efeito de lote significativo (Fase 3, acima). Se o lote desloca a
corrente de forma parecida nos 3 estágios da mesma réplica (mesmo eletrodo físico), calcular a
*diferença* entre estágios deveria cancelar boa parte desse deslocamento — isolando o que
sobra: o efeito real da ligação com o alvo.

### O que foi testado

1. Reestruturar os dados: 1 linha por **amostra + replicata** (não mais por medição/estágio),
   combinando os 3 estágios da mesma réplica.
2. Suavização (Savitzky–Golay) antes de extrair descritores.
3. Descritores mais ricos por estágio: pico, mínimo, amplitude, **área sob a curva**,
   **potencial do pico**, **largura do pico**.
4. Features de diferença entre estágios: `delta_peak`, `delta_auc`, `delta_potencial`, e a
   **curva de diferença completa** (corrente do estágio 2 menos estágio 1, ponto a ponto).
5. Dois modelos novos: **SVM (RBF)** e **Gradient Boosting**, além dos 3 já usados.
6. Validação mais robusta: 10 sementes de split treino/teste (em vez de uma só), reportando
   média ± desvio.
7. Otimização do limiar de decisão (estatística de Youden) em vez do corte fixo 0,5.


In [ ]:
import sys
sys.path.insert(0, "src")
from advanced_modeling import build_stage_diff_dataset, run_multiseed_comparison, summarize, detailed_report_best_config

stage_dataset = build_stage_diff_dataset(measurements, keep_flags=("PASS",))
print(f"Dataset por amostra+replicata: {len(stage_dataset)} linhas "
      f"(de até {measurements.groupby(['sample_id','replicate_id']).ngroups} réplicas possíveis)")
stage_dataset.head(3)


**Nota sobre a redução de linhas**: essa abordagem exige os 3 estágios completos (com
`qc_flag = PASS`) na mesma réplica — algumas réplicas têm 1 ou 2 estágios reprovados e são
descartadas. É um trade-off real: menos dado, mas isolando o sinal certo.


In [ ]:
seeds = list(range(10))
results_summary = []
for fs in ["stage_diff_summary", "stage_diff_full_curve"]:
    results_summary.append(run_multiseed_comparison(stage_dataset, fs, seeds))
comparison = summarize(pd.concat(results_summary, ignore_index=True))
comparison[["feature_set","model","balanced_accuracy_mean","balanced_accuracy_std","roc_auc_mean","sensitivity_mean","specificity_mean","n_seeds"]]


### Checagem de vazamento de dados (obrigatória antes de comemorar um salto grande)

Um salto de ~57% para ~94% de balanced accuracy é grande o suficiente pra merecer suspeita.
Antes de aceitar o resultado, checamos duas coisas: (1) se `sensor_lot` determina o rótulo
de forma direta (vazamento via confundidor), e (2) se uma única feature explica tudo sozinha
(o que sugeriria um atalho, não um padrão real).


In [ ]:
print("=== contamination_status por lote (nenhum lote é 100% puro) ===")
print(pd.crosstab(stage_dataset["sensor_lot"], stage_dataset["contamination_status"]))
print()
print("=== features isoladas mais correlacionadas com o rótulo (nenhuma é ~1.0) ===")
corr = stage_dataset.drop(columns=["sample_id","replicate_id","sensor_lot","target_bacterium"]).corr()["contamination_status"].abs().sort_values(ascending=False)
print(corr.head(8))


Nenhum lote é puro (todos têm mistura de contaminadas/não) e a feature isolada mais forte
tem correlação de ~0,70 — forte, mas longe de 1,0. O modelo combina ~130 pontos correlacionados
da curva de diferença pra chegar em alta separação. **Não é vazamento — é o formato da curva de
diferença carregando mais informação do que qualquer estatística-resumo isolada.**

### Resultado detalhado da melhor configuração


In [ ]:
best_row = comparison.iloc[0]
detail = detailed_report_best_config(stage_dataset, best_row["feature_set"], best_row["model"], seed=42)

cm = np.array(detail["confusion_matrix_default"])
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Greens")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Não contaminada", "Contaminada"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Não contaminada", "Contaminada"])
ax.set_xlabel("Predito"); ax.set_ylabel("Real")
ax.set_title(f"{detail['model']} + {detail['feature_set']} (seed 42)")
plt.tight_layout()
plt.show()

print(f"Balanced accuracy: {detail['balanced_accuracy_default']:.1%} | ROC-AUC: {detail['roc_auc']:.3f}")
print(f"Treino: {detail['n_train']} réplicas | Teste: {detail['n_test']} réplicas | Features: {detail['n_features']}")
print(f"Limiar ótimo (Youden): {detail['threshold_opt']:.3f} (corte padrão seria 0.5)")


### Conclusão da melhoria

| Abordagem | Melhor modelo | Balanced accuracy | ROC-AUC |
|---|---|---|---|
| Medição isolada (baseline) | Random Forest | ~57% | ~0,58 |
| Diferença entre estágios — resumo (pico/AUC) | Reg. Logística | ~79% | ~0,86 |
| **Diferença entre estágios — curva completa** | **Reg. Logística** | **~94%** | **~0,98** |

A hipótese central do sensor (a *mudança* de sinal entre estágios é o que importa, não o
estágio isolado) se confirma fortemente nos dados — e de quebra, cancela boa parte do efeito
de lote identificado antes. Isso muda a conclusão do case de "sinal fraco, precisa melhorar"
para "sinal forte, desde que comparado corretamente entre estágios do mesmo sensor".

**Limitações que permanecem**: (i) essa validação usa 10 splits treino/teste diferentes mas
ainda é um único dataset — validação externa (novo lote, novo operador) é o próximo teste
natural; (ii) 502 de 600 réplicas possíveis são usadas (perde-se réplicas com qualquer estágio
fora de `qc_flag=PASS`); (iii) Regressão Logística superou Random Forest/SVM/Gradient Boosting
aqui — plausível dado ~150 features e algumas centenas de réplicas, onde um modelo linear bem
regularizado generaliza melhor que modelos mais flexíveis.

**Aplicação a dados novos**: qualquer novo lote com o mesmo schema de colunas pode ser
adicionado à pasta de entrada e reprocessado rodando `data_preparation.py`, `ingest_pipeline.py`
e `advanced_modeling.py` em sequência — nenhuma estrutura precisa ser reconstruída manualmente.
